# Asignación de coordenadas – Método 2 (KDE + DBSCAN)
Genera un archivo de coordenadas únicas con su asignación a hubs M2 a 50 / 100 / 150 millas.
- **Input hubs**: `../5. merge dbscan/results/final_results.csv`
- **Input cargas**: `../2. cleaning_loads/parquetloadsfull.parquet`
- **Output**: `coords_assign_m2.parquet`

### Metadata del análisis
| Campo | Valor |
|---|---|
| N registros | 57,573,489 |
| Ventana de fechas | Feb–Ago 2025 |
| Tipo de validación | Within-sample |
| Input hubs | `final_results_optimized.csv` — 188 hubs M2 orgánicos (post-optimización, sin NJ/CT/RI/DE) |

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from tqdm.auto import tqdm

/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Hubs M2

In [ ]:
hubs_raw = pd.read_csv('final_results_optimized.csv')

rows = []
for _, r in hubs_raw.dropna(subset=['lat', 'lng']).iterrows():
    rows.append({
        'hub_id':   str(r['hub_id']),
        'hub_name': r.get('new_name') or str(r['hub_id']),
        'lat':      float(r['lat']),
        'lng':      float(r['lng']),
    })

hubs = pd.DataFrame(rows).reset_index(drop=True)
print(f'Hubs M2: {len(hubs)}')

## 2. Coordenadas únicas

In [3]:
raw = (
    pl.scan_parquet('../2. cleaning_loads/parquetloadsfull.parquet')
    .select(['LatOrigin', 'LngOrigin', 'LatDestination', 'LngDestination'])
    .collect()
)

orig = raw.select([
    pl.col('LatOrigin').cast(pl.Float64).round(6).alias('lat'),
    pl.col('LngOrigin').cast(pl.Float64).round(6).alias('lng'),
]).drop_nulls()

dest = raw.select([
    pl.col('LatDestination').cast(pl.Float64).round(6).alias('lat'),
    pl.col('LngDestination').cast(pl.Float64).round(6).alias('lng'),
]).drop_nulls()

coords = (
    pl.concat([orig, dest])
    .unique()
    .to_pandas()
    .reset_index(drop=True)
)

del raw, orig, dest
print(f'Coordenadas unicas: {len(coords):,}')

Coordenadas unicas: 738,827


## 3. Asignación vectorizada por batches

In [4]:
def assign_batched(coords_df, hubs_df, batch_size=5_000, radii=(50, 100, 150)):
    n        = len(coords_df)
    hub_lats = np.radians(hubs_df['lat'].values)
    hub_lngs = np.radians(hubs_df['lng'].values)
    hub_ids  = hubs_df['hub_id'].values

    near_hub = np.empty(n, dtype=object)
    hubs_r   = {r: [None] * n for r in radii}

    for start in tqdm(range(0, n, batch_size)):
        end  = min(start + batch_size, n)
        blat = np.radians(coords_df['lat'].values[start:end, np.newaxis])
        blng = np.radians(coords_df['lng'].values[start:end, np.newaxis])

        dlat = hub_lats - blat
        dlng = hub_lngs - blng
        a    = (np.sin(dlat / 2) ** 2
                + np.cos(blat) * np.cos(hub_lats) * np.sin(dlng / 2) ** 2)
        dist = 3958.8 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

        near_hub[start:end] = hub_ids[dist.argmin(axis=1)]
        for r in radii:
            mask = dist <= r
            for i in range(end - start):
                hubs_r[r][start + i] = list(hub_ids[mask[i]])

    out = coords_df.copy()
    out['near_hub_m2'] = near_hub
    for r in radii:
        out[f'hubs_{r}m_m2'] = hubs_r[r]
    return out


In [5]:
coords_assigned = assign_batched(coords, hubs)
coords_assigned.head()

100%|████████████████████████████████████████████████████████████████████████████████| 148/148 [00:30<00:00,  4.80it/s]


,lat,lng,near_hub_m2,hubs_50m_m2,hubs_100m_m2,hubs_150m_m2
0,37.308760,-79.901985,71.0,[71.0],"[71.0, 201.0, C181]","[93.0, 186.0, 71.0, 201.0, C141, C181]"
1,35.186402,-84.828057,65.0,[65.0],"[36.0, 63.0, 65.0]","[1.0, 36.0, 179.0, 63.0, 65.0, 81.0, C110]"
2,34.473400,-82.656997,158.0,[],"[158.0, 66.0, C110]","[36.0, 93.0, 158.0, 66.0, 81.0, C110]"
3,41.712752,-93.591708,38.0,[38.0],"[38.0, 72.0]","[38.0, 72.0, 109.0, C111, C112, C126]"
4,39.768990,-104.765730,16.0,[16.0],"[16.0, C102]","[16.0, 221.0, C102, C149]"


## 4. Guardar

In [6]:
out = pl.from_pandas(coords_assigned)
out.write_parquet('coords_assign_m2.parquet')
print(f'Guardado: coords_assign_m2.parquet  ({len(out):,} filas)')
print(out.schema)
print(out.head())

Guardado: coords_assign_m2.parquet  (738,827 filas)
Schema([('lat', Float64), ('lng', Float64), ('near_hub_m2', String), ('hubs_50m_m2', List(String)), ('hubs_100m_m2', List(String)), ('hubs_150m_m2', List(String))])
shape: (5, 6)
┌───────────┬────────────┬─────────────┬─────────────┬───────────────────┬─────────────────────────┐
│ lat       ┆ lng        ┆ near_hub_m2 ┆ hubs_50m_m2 ┆ hubs_100m_m2      ┆ hubs_150m_m2            │
│ ---       ┆ ---        ┆ ---         ┆ ---         ┆ ---               ┆ ---                     │
│ f64       ┆ f64        ┆ str         ┆ list[str]   ┆ list[str]         ┆ list[str]               │
╞═══════════╪════════════╪═════════════╪═════════════╪═══════════════════╪═════════════════════════╡
│ 37.30876  ┆ -79.901985 ┆ 71.0        ┆ ["71.0"]    ┆ ["71.0", "201.0", ┆ ["93.0", "186.0", …     │
│           ┆            ┆             ┆             ┆ "C181"]           ┆ "C181"]                 │
│ 35.186402 ┆ -84.828057 ┆ 65.0        ┆ ["65.0"]    ┆ ["36.0"